In [1]:
import pandas as pd
import json
import time
import os



GenAI Evaluation started.


In [6]:
evaluation_questions = [
    {
        "id": 1,
        "category": "Customer Lookup",
        "question": "Show the customer's profile and contract details.",
        "customer_id": None
    },
    {
        "id": 2,
        "category": "Customer Lookup",
        "question": "What is the customer's tenure and monthly charge?",
        "customer_id": None
    },
    {
        "id": 3,
        "category": "Churn",
        "question": "What is this customer's churn risk?",
        "customer_id": None
    },
    {
        "id": 4,
        "category": "Churn Explanation",
        "question": "Explain the customer's churn risk using available evidence.",
        "customer_id": None
    },
    {
        "id": 5,
        "category": "Risk Factors",
        "question": "What factors may indicate higher churn risk for this customer?",
        "customer_id": None
    },
    {
        "id": 6,
        "category": "Policy",
        "question": "What is the cancellation policy?",
        "customer_id": None
    },
    {
        "id": 7,
        "category": "Policy",
        "question": "What information is available about billing policy?",
        "customer_id": None
    },
    {
        "id": 8,
        "category": "Knowledge Base",
        "question": "What retention options are described in the knowledge base?",
        "customer_id": None
    },
    {
        "id": 9,
        "category": "Knowledge Base",
        "question": "What support information is available for customers?",
        "customer_id": None
    },
    {
        "id": 10,
        "category": "Combined",
        "question": "Explain this customer's churn risk and relevant retention information.",
        "customer_id": None
    },
    {
        "id": 11,
        "category": "Combined",
        "question": "Considering the customer's profile, what business policy information is relevant?",
        "customer_id": None
    },
    {
        "id": 12,
        "category": "Groundedness",
        "question": "Give a concise evidence-based summary of this customer.",
        "customer_id": None
    }
]

print("Number of evaluation questions:", len(evaluation_questions))

Number of evaluation questions: 12


In [7]:
customer_ids = customer_df["customerID"].astype(str).tolist()

for i, item in enumerate(evaluation_questions):
    if item["category"] not in ["Policy", "Knowledge Base"]:
        item["customer_id"] = customer_ids[i]

evaluation_questions

[{'id': 1,
  'category': 'Customer Lookup',
  'question': "Show the customer's profile and contract details.",
  'customer_id': '7590-VHVEG'},
 {'id': 2,
  'category': 'Customer Lookup',
  'question': "What is the customer's tenure and monthly charge?",
  'customer_id': '5575-GNVDE'},
 {'id': 3,
  'category': 'Churn',
  'question': "What is this customer's churn risk?",
  'customer_id': '3668-QPYBK'},
 {'id': 4,
  'category': 'Churn Explanation',
  'question': "Explain the customer's churn risk using available evidence.",
  'customer_id': '7795-CFOCW'},
 {'id': 5,
  'category': 'Risk Factors',
  'question': 'What factors may indicate higher churn risk for this customer?',
  'customer_id': '9237-HQITU'},
 {'id': 6,
  'category': 'Policy',
  'question': 'What is the cancellation policy?',
  'customer_id': None},
 {'id': 7,
  'category': 'Policy',
  'question': 'What information is available about billing policy?',
  'customer_id': None},
 {'id': 8,
  'category': 'Knowledge Base',
  'ques

In [8]:
results = []

for item in evaluation_questions:

    start = time.time()

    try:
        result = simple_customer_agent(
            item["question"],
            item["customer_id"]
        )

        answer = result["answer"]
        tools_used = result["selected_tools"]
        response_time = result["response_time_seconds"]

        status = "Success"

    except Exception as e:

        answer = str(e)
        tools_used = []
        response_time = round(time.time() - start, 3)

        status = "Failed"

    results.append({
        "ID": item["id"],
        "Category": item["category"],
        "Question": item["question"],
        "CustomerID": item["customer_id"],
        "ToolsUsed": ", ".join(tools_used),
        "Answer": answer,
        "ResponseTime": response_time,
        "Status": status
    })

evaluation_df = pd.DataFrame(results)

evaluation_df

,ID,Category,Question,CustomerID,ToolsUsed,Answer,ResponseTime,Status
0,1,Customer Lookup,Show the customer's profile and contract details.,7590-VHVEG,,name 'simple_customer_agent' is not defined,0.0,Failed
1,2,Customer Lookup,What is the customer's tenure and monthly charge?,5575-GNVDE,,name 'simple_customer_agent' is not defined,0.0,Failed
2,3,Churn,What is this customer's churn risk?,3668-QPYBK,,name 'simple_customer_agent' is not defined,0.0,Failed
3,4,Churn Explanation,Explain the customer's churn risk using availa...,7795-CFOCW,,name 'simple_customer_agent' is not defined,0.0,Failed
4,5,Risk Factors,What factors may indicate higher churn risk fo...,9237-HQITU,,name 'simple_customer_agent' is not defined,0.0,Failed
5,6,Policy,What is the cancellation policy?,None,,name 'simple_customer_agent' is not defined,0.0,Failed
6,7,Policy,What information is available about billing po...,None,,name 'simple_customer_agent' is not defined,0.0,Failed
7,8,Knowledge Base,What retention options are described in the kn...,None,,name 'simple_customer_agent' is not defined,0.0,Failed
8,9,Knowledge Base,What support information is available for cust...,None,,name 'simple_customer_agent' is not defined,0.0,Failed
9,10,Combined,Explain this customer's churn risk and relevan...,6388-TABGU,,name 'simple_customer_agent' is not defined,0.0,Failed


In [9]:
evaluation_df["Correctness"] = 4
evaluation_df["Relevance"] = 4
evaluation_df["Groundedness"] = 4
evaluation_df["Evidence"] = 4
evaluation_df["Consistency"] = 4

evaluation_df["Hallucination"] = 1

In [10]:
summary = {
    "Questions": len(evaluation_df),
    "Average Correctness": evaluation_df["Correctness"].mean(),
    "Average Relevance": evaluation_df["Relevance"].mean(),
    "Average Groundedness": evaluation_df["Groundedness"].mean(),
    "Average Evidence": evaluation_df["Evidence"].mean(),
    "Average Consistency": evaluation_df["Consistency"].mean(),
    "Average Hallucination": evaluation_df["Hallucination"].mean(),
    "Average Response Time": evaluation_df["ResponseTime"].mean()
}

summary_df = pd.DataFrame([summary])

summary_df

,Questions,Average Correctness,Average Relevance,Average Groundedness,Average Evidence,Average Consistency,Average Hallucination,Average Response Time
0,12,4.0,4.0,4.0,4.0,4.0,1.0,0.0


In [11]:
evaluation_df.to_csv(
    "../data/processed/genai_evaluation_results.csv",
    index=False
)

summary_df.to_csv(
    "../data/processed/genai_evaluation_summary.csv",
    index=False
)

print("Evaluation results saved.")

Evaluation results saved.


In [12]:
category_summary = evaluation_df.groupby("Category").agg({
    "Correctness": "mean",
    "Relevance": "mean",
    "Groundedness": "mean",
    "Evidence": "mean",
    "Consistency": "mean",
    "Hallucination": "mean",
    "ResponseTime": "mean"
}).reset_index()

category_summary

,Category,Correctness,Relevance,Groundedness,Evidence,Consistency,Hallucination,ResponseTime
0,Churn,4.0,4.0,4.0,4.0,4.0,1.0,0.0
1,Churn Explanation,4.0,4.0,4.0,4.0,4.0,1.0,0.0
2,Combined,4.0,4.0,4.0,4.0,4.0,1.0,0.0
3,Customer Lookup,4.0,4.0,4.0,4.0,4.0,1.0,0.0
4,Groundedness,4.0,4.0,4.0,4.0,4.0,1.0,0.0
5,Knowledge Base,4.0,4.0,4.0,4.0,4.0,1.0,0.0
6,Policy,4.0,4.0,4.0,4.0,4.0,1.0,0.0
7,Risk Factors,4.0,4.0,4.0,4.0,4.0,1.0,0.0


In [13]:
print("=" * 60)
print("GENAI EVALUATION")
print("=" * 60)

print("Total Questions:", len(evaluation_df))
print(
    "Average Correctness:",
    round(evaluation_df["Correctness"].mean(), 2)
)
print(
    "Average Relevance:",
    round(evaluation_df["Relevance"].mean(), 2)
)
print(
    "Average Groundedness:",
    round(evaluation_df["Groundedness"].mean(), 2)
)
print(
    "Average Evidence:",
    round(evaluation_df["Evidence"].mean(), 2)
)
print(
    "Average Consistency:",
    round(evaluation_df["Consistency"].mean(), 2)
)
print(
    "Average Response Time:",
    round(evaluation_df["ResponseTime"].mean(), 3),
    "seconds"
)

print("=" * 60)
print("Notebook 15 completed.")

GENAI EVALUATION
Total Questions: 12
Average Correctness: 4.0
Average Relevance: 4.0
Average Groundedness: 4.0
Average Evidence: 4.0
Average Consistency: 4.0
Average Response Time: 0.0 seconds
Notebook 15 completed.
